<a href="https://colab.research.google.com/github/48kashviyye/Skill-Training/blob/main/Amazon_AI_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
%pip install -q playwright google-genai python-dotenv

In [14]:
!playwright install chromium

In [15]:
import os
from getpass import getpass
os.environ["GEMINI_API_KEY"] = getpass("Enter Gemini API Key: ")

Enter Gemini API Key: ··········


In [16]:
import asyncio
from google import genai
from google.genai import types
from playwright.async_api import async_playwright

client = genai.Client()

print("All libraries imported successfully!")

All libraries imported successfully!


In [17]:
async def start_browser():

    global pw, browser, context, page

    pw = await async_playwright().start()

    browser = await pw.chromium.launch(
        headless=True
    )

    context = await browser.new_context()

    page = await context.new_page()

    print("Chrome browser is ready!")

await start_browser()

Chrome browser is ready!


In [18]:
!playwright install-deps chromium

Installing dependencies...
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:3 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:4 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libasound2t64 is already the newest version (1.2.11-1ubuntu0.3).
libatk-bridge2.0-0t64 is already the newest version (2.52.0-1build1).
libatk1.

In [19]:
print("Browser connected:", browser.is_connected())

Browser connected: True


In [20]:
async def open_amazon():

    print("Opening Amazon India...")

    await page.goto(
        "https://www.amazon.in",
        wait_until="domcontentloaded",
        timeout=60000
    )

    print("Amazon opened successfully!")

    print("Website title:", await page.title())
    print("Website URL:", page.url)


await open_amazon()

Opening Amazon India...
Amazon opened successfully!
Website title: Online Shopping site in India: Shop Online for Mobiles, Books, Watches, Shoes and More - Amazon.in
Website URL: https://www.amazon.in/


In [22]:
async def search_product(keyword: str):

    print(f"Searching for: {keyword}")

    try:
        search_box = page.locator("#twotabsearchtextbox")

        await search_box.wait_for(
            state="visible",
            timeout=15000
        )

        await search_box.fill(keyword)

        await page.locator(
            "#nav-search-submit-button"
        ).click()

        await page.wait_for_selector(
            "[data-component-type='s-search-result']",
            timeout=20000
        )

        print("Product search completed!")

        return True

    except Exception as e:
        print("Search failed:", str(e))
        return False


result = await search_product("wireless bluetooth headphones under 1000")

print("Search status:", result)

Searching for: wireless bluetooth headphones under 1000
Product search completed!
Search status: True


In [25]:
%pip install playwright google-genai python-dotenv
!playwright install chromium
import asyncio
from dotenv import load_dotenv
from playwright.async_api import async_playwright
from google import genai
from google.genai import types

load_dotenv()
client = genai.Client()

pw = await async_playwright().start()
browser = await pw.chromium.launch(headless=True, slow_mo=700)
context = await browser.new_context(
    user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
)
page = await context.new_page()
print("Chrome is ready!")
from google.genai.errors import ServerError

async def open_amazon() -> str:
    """Opens Amazon India (amazon.in) in the Chrome browser."""
    print("  [CHROME] Opening amazon.in...")
    await page.goto("https://www.amazon.in", wait_until="domcontentloaded")
    return "Amazon India homepage is open."

async def search_product(keyword: str) -> str:
    """Searches for a product keyword on Amazon India."""
    print(f"  [CHROME] Searching Amazon.in for: '{keyword}'")
    await page.wait_for_selector("#twotabsearchtextbox", timeout=10000)
    await page.fill("#twotabsearchtextbox", keyword)
    await page.click("#nav-search-submit-button")
    await page.wait_for_selector("[data-component-type='s-search-result']", timeout=10000)
    return f"Search results loaded for '{keyword}'."

async def get_top_products(limit: int = 5) -> str:
    """Extracts the titles and prices of the top search results on Amazon India."""
    print(f"  [CHROME] Reading top {limit} products...")
    cards = await page.locator("[data-component-type='s-search-result']").all()
    results = []

    for card in cards[:limit]:
        title_loc = card.locator("h2 span").first
        price_loc = card.locator(".a-price .a-offscreen").first

        title = (await title_loc.inner_text()).strip() if await title_loc.count() > 0 else "No title"
        price = (await price_loc.inner_text()).strip() if await price_loc.count() > 0 else "Price unavailable"
        results.append(f"- {title} | {price}")

    return "\n".join(results)

tools = [open_amazon, search_product, get_top_products]

goal = """
Open Amazon India, search for 'wireless mouse under 1000',
check the top 5 products, and recommend the best value option from the list.
"""

# Fallback list so a 503 spike never stalls your live class
models_to_try = ["gemini-3.8-flash", "gemini-3.5-flash", "gemini-3.5-flash-lite"]

for model_name in models_to_try:
    try:
        print(f"Trying {model_name}...")
        chat = client.aio.chats.create(
            model=model_name,
            config=types.GenerateContentConfig(tools=tools, temperature=0.0)
        )
        response = await chat.send_message(goal)
        print(f"\nAgent Recommendation (via {model_name}):\n", response.text)
        break
    except ServerError as e:
        print(f"  [!] {model_name} busy ({e.code}), switching to fallback...")
await browser.close()
await pw.stop()

Chrome is ready!
Trying gemini-3.8-flash...
  [!] gemini-3.8-flash busy (503), switching to fallback...
Trying gemini-3.5-flash...
  [!] gemini-3.5-flash busy (503), switching to fallback...
Trying gemini-3.5-flash-lite...
  [CHROME] Opening amazon.in...
  [CHROME] Searching Amazon.in for: 'wireless mouse under 1000'
  [!] gemini-3.5-flash-lite busy (503), switching to fallback...
